# 01 · Explore the CMI data

Phase 1, Nov 3–4. Goal: understand the data well enough to explain it in two sentences.

We only use the **motion** columns (`acc_*`, `rot_*`), because that's all the MPU6050 can measure. See [HARDWARE.md](../HARDWARE.md).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = "../data/cmi-detect-behavior-with-sensor-data"
ACC = ["acc_x", "acc_y", "acc_z"]
ROT = ["rot_w", "rot_x", "rot_y", "rot_z"]
LABELS = ["sequence_id", "sequence_counter", "subject", "sequence_type", "orientation", "phase", "gesture"]

df = pd.read_csv(f"{DATA}/train.csv", usecols=LABELS + ACC + ROT)
demo = pd.read_csv(f"{DATA}/train_demographics.csv")
df.shape

## 1. What's in it

One row is one sensor reading. A **sequence** is one recording of one person doing one gesture. `sequence_type` says whether the gesture is BFRB-like (`Target`) or ordinary (`Non-Target`).

In [ ]:
seqs = df.groupby("sequence_id").agg(
    subject=("subject", "first"),
    sequence_type=("sequence_type", "first"),
    gesture=("gesture", "first"),
    orientation=("orientation", "first"),
    rows=("sequence_counter", "size"),
    gesture_rows=("phase", lambda p: (p == "Gesture").sum()),
)
print("people:", seqs.subject.nunique())
print("sequences:", len(seqs))
seqs.sequence_type.value_counts()

### Which gestures, and how many of each

In [ ]:
seqs.groupby(["sequence_type", "gesture"]).size().sort_values(ascending=False)

### Body positions

Every gesture was done in four positions. The model has to work in all of them.

In [ ]:
pd.crosstab(seqs.orientation, seqs.sequence_type)

### About the people

In [ ]:
print("adults:", (demo.adult_child == 1).sum(), " children:", (demo.adult_child == 0).sum())
print("right-handed:", (demo.handedness == 1).sum(), " left-handed:", (demo.handedness == 0).sum())
demo.age.describe()

## 2. How long is a gesture?

Each sequence has a **Transition** part (moving the hand into place) and then the **Gesture** part. Lengths are in rows (readings), because the data has no timestamps.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
seqs.rows.clip(upper=200).hist(bins=50, ax=ax[0])
ax[0].set(title="Whole sequence", xlabel="rows (clipped at 200)", ylabel="sequences")
seqs.gesture_rows.clip(upper=150).hist(bins=50, ax=ax[1])
ax[1].set(title="Gesture part only", xlabel="rows (clipped at 150)")
plt.tight_layout()
seqs[["rows", "gesture_rows"]].describe().round(1)

## 3. Data quality

Acceleration is in m/s² with gravity included, so at rest its total should be about 9.8. Some rows have no orientation (`rot`) values.

In [ ]:
acc_total = (df[ACC] ** 2).sum(axis=1) ** 0.5
print("median total acceleration:", round(acc_total.median(), 2), "m/s²")
missing = df[ROT].isna().any(axis=1)
print("rows missing rot:", missing.sum(), " in sequences:", df.loc[missing, "sequence_id"].nunique())

## 4. A BFRB-like gesture next to an ordinary one

Same person, same body position. The shaded part is the gesture itself.

In [ ]:
BFRB = "Forehead - pull hairline"
ORDINARY = "Wave hello"
POSITION = "Seated Straight"

# pick a person who did both gestures in this position
both = seqs[seqs.orientation == POSITION].groupby("subject").gesture.agg(set)
subject = both[both.apply(lambda g: {BFRB, ORDINARY} <= g)].index[0]

def plot_sequence(gesture, axes):
    sid = seqs[(seqs.subject == subject) & (seqs.gesture == gesture) & (seqs.orientation == POSITION)].index[0]
    s = df[df.sequence_id == sid].reset_index(drop=True)
    start = s.index[s.phase == "Gesture"][0]
    for ax, cols, title in ((axes[0], ACC, "acceleration (m/s²)"), (axes[1], ROT, "orientation (quaternion)")):
        s[cols].plot(ax=ax)
        ax.axvspan(start, len(s) - 1, color="grey", alpha=0.15)
        ax.set(title=f"{gesture}: {title}", xlabel="row")
        ax.legend(loc="upper left", fontsize=8)

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
plot_sequence(BFRB, axes[:, 0])
plot_sequence(ORDINARY, axes[:, 1])
plt.suptitle(f"{subject}, {POSITION}")
plt.tight_layout()

**What to notice**

- The wave is fast and shaky during the gesture; the hair pull is mostly still, with the hand held up at the head. Holding still in an odd position is a big clue for BFRB-like gestures.
- The quaternion lines sometimes jump to the mirror value (e.g. `rot_y` flipping between about +0.8 and -0.8) while the wrist barely moves. That's because `q` and `-q` describe the same orientation. Raw quaternions are a bad model input for this reason, which is another reason to use the derived features from [HARDWARE.md](../HARDWARE.md) (tilt, gravity-free acceleration, rotation speed).
- A few sequences have **no Gesture rows** at all (see the minimum of 0 in section 2). Drop them before training.

## 5. Your summary

Write the two-sentence explanation of the data here, in your own words:

1. …
2. …